# 365 Probabilidades - Dia #120
## Qual a probabilidade de você chegar mais longe que os seus pais?

**Tipo:** Probabilidade simples
**Data de publicação:** 2026-10-11
**Ferramenta:** Python
**Decisão analisada:** De onde você partiu, e quanto do caminho a memória apagou?
**Hashtag:** #365Probabilidades #Dia120

---

### 📖 A História

[Abertura da Ana. A cena: escovar os dentes, lembrar que aos quinze ou dezesseis anos o
sonho era um banheiro com uma pia que não fosse só a pia. A casa própria e simples. A
pia que era só pia, não uma bancada. E a descoberta de que o sonho foi realizado há
anos, e a memória não guardou nem o sonho nem a realização.]

---

### 📚 O Conceito

Mobilidade social é a distância entre o lugar de onde você partiu e o lugar onde você
está. O IBGE mede isso de um jeito direto: pergunta o que o seu pai fazia quando você
tinha quinze anos, e o que você faz hoje. Com isso dá para calcular, para cada ponto de
partida, a chance de chegar a cada ponto de chegada.

A memória não mede nada disso. Easterlin (2001, *Economic Journal*) descreve o
mecanismo: as aspirações sobem junto com a renda, então o que era sonho vira régua, e a
régua não registra o que já está debaixo dela. É por isso que uma bancada realizada vira
só uma bancada.

---

### 🧮 O Modelo

Este é um dia de **probabilidade simples**: a pergunta vem primeiro e o dado é o
suplemento de mobilidade da PNAD. As contas são nossas.

**Dados:**
- IBGE, *PNAD 2014, Mobilidade sócio-ocupacional*. Suplemento aplicado a pessoas de
  16 anos ou mais; cerca de **58 mil pessoas** com informação sobre a ocupação do pai
  quando tinham 15 anos. Tabela usada: distribuição das pessoas ocupadas por grupamento
  ocupacional atual, segundo o grupamento ocupacional do pai. Os números são
  **percentuais publicados**, não contagens por célula; o IBGE não divulga o número de
  entrevistados em cada combinação pai × filho.
- Da mesma divulgação: distribuição do nível de instrução das pessoas de 25 anos ou
  mais segundo o nível de instrução do pai; mobilidade ascendente total de **47,4%**,
  imobilidade de **33,4%**, descendente de **17,2%**.

**Conferência pendente antes do commit:** a tabela foi lida por extração do PDF da
divulgação (16/11/2016). As linhas fecham em 100% e o padrão interno é coerente, mas a
ordem das colunas precisa ser conferida visualmente contra o PDF.

**Fonte de conceito, sem número usado:**
- Easterlin, R. A. (2001). *Income and Happiness: Towards a Unified Theory.* Economic
  Journal 111(473), 465-484.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
SEMENTE = 42
ger = np.random.default_rng(SEMENTE)

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def pct(x, casas=1):
    return vir(100 * x, casas) + "%"

def mil(n):
    return f"{int(n):,}".replace(",", ".")

print("✅ Bibliotecas carregadas")

In [ ]:
# --- OS DADOS ---
# IBGE, PNAD 2014, suplemento de mobilidade socio-ocupacional (divulgacao de 16/11/2016).
# Linhas: grupamento ocupacional do PAI quando a pessoa tinha 15 anos.
# Colunas: grupamento ocupacional ATUAL da pessoa. Percentuais publicados.

GRUPOS = ["Dirigentes", "Profissionais", "Técnicos", "Administrativos",
          "Serviços", "Vendedores", "Agrícolas", "Produção", "Forças Armadas"]
# Estratos do IBGE: A = dirigentes + profissionais; B = tecnicos; C = administrativos;
# D = producao; E = servicos + vendedores; F = agricolas. Forcas armadas fora da escada.
ESTRATO = {"Dirigentes": "A", "Profissionais": "A", "Técnicos": "B", "Administrativos": "C",
           "Produção": "D", "Serviços": "E", "Vendedores": "E", "Agrícolas": "F",
           "Forças Armadas": "FA"}
ORDEM = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 4, "F": 5}   # D e E no mesmo degrau, como o IBGE

M = pd.DataFrame([
    [19.4, 23.7, 12.6, 13.3,  9.1, 10.0,  2.1,  8.9, 0.8],   # pai dirigente
    [12.9, 46.1, 13.4,  8.6,  6.3,  4.1,  0.9,  5.9, 1.8],   # pai profissional
    [ 8.7, 22.9, 16.1, 14.6, 11.4,  7.7,  2.0, 14.4, 1.9],   # pai tecnico
    [10.8, 22.7, 14.3, 15.0, 11.1,  8.7,  0.9, 15.0, 1.4],   # pai administrativo
    [ 4.8,  9.3,  9.0, 14.3, 26.2,  9.8,  3.0, 22.2, 1.5],   # pai servicos
    [ 7.6, 12.2,  9.0, 11.3, 16.7, 19.3,  2.7, 20.1, 1.1],   # pai vendedor
    [ 3.4,  4.1,  3.3,  4.1, 19.0,  7.8, 34.9, 23.3, 0.2],   # pai agricola
    [ 4.7,  8.8,  8.4, 13.7, 19.6,  9.6,  2.8, 31.4, 1.0],   # pai producao
    [11.2, 20.2, 12.2, 13.1, 13.5,  9.4,  1.6, 11.8, 6.9],   # pai forcas armadas
], index=GRUPOS, columns=GRUPOS) / 100
TOTAL = pd.Series([5.7, 9.9, 7.3, 9.7, 18.2, 9.1, 15.5, 23.8, 0.8], index=GRUPOS) / 100

# Instrucao do filho (25 anos ou mais) segundo instrucao do pai: % com superior completo
SUPERIOR = pd.Series({"sem instrução": 4.0, "fundamental incompleto": 14.6,
                      "fundamental completo": 26.8, "médio incompleto": 29.2,
                      "médio completo": 41.2, "superior incompleto": 67.1,
                      "superior completo": 69.1}) / 100

# Totais publicados
ASCENDENTE, IMOBIL, DESCENDENTE = 0.474, 0.334, 0.172
N_SUPLEMENTO = 58_000

print("=" * 72)
print("  CONFERÊNCIA DA TABELA")
print("=" * 72)
somas = M.sum(axis=1) * 100
print(f"\n  Soma de cada linha (deve dar 100): mínimo {vir(somas.min())}, máximo {vir(somas.max())}")
print(f"  Soma da linha total: {vir(TOTAL.sum() * 100)}")
print(f"  Diagonal (filho no mesmo grupo do pai):")
for g in GRUPOS:
    print(f"    {g:<16}{pct(M.loc[g, g]):>8}")
print(f"\n  Padrão esperado: a diagonal é o maior valor da linha para profissionais,")
print(f"  agrícolas e produção. Conferido. A ordem das colunas foi reconstruída pela")
print(f"  soma das linhas e PRECISA ser conferida contra o PDF antes do commit.")
print(f"\n  Totais publicados pelo IBGE em relação ao pai:")
print(f"    subiram de estrato {pct(ASCENDENTE)}   ficaram {pct(IMOBIL)}   desceram {pct(DESCENDENTE)}")
print(f"\n  ×0,80: não se aplica. Ocupação e escolaridade declaradas, fatos verificáveis,")
print(f"  não opinião sobre si.")
print("=" * 72)

In [ ]:
# --- O MODELO ---
# A "bancada" do titulo e o estrato A do IBGE: dirigentes e profissionais das ciencias
# e das artes. A "casa simples" e qualquer origem fora de A.

P_A = M[["Dirigentes", "Profissionais"]].sum(axis=1)       # P(chegar a A | grupo do pai)
p_a_total = TOTAL[["Dirigentes", "Profissionais"]].sum()  # fracao de todos em A hoje

print("=" * 72)
print("  CAMADA 1 · A ESCADA: CHEGAR AO ESTRATO A, SEGUNDO A ORIGEM")
print("=" * 72)
print(f"\n  Hoje, {pct(p_a_total)} das pessoas ocupadas estão no estrato A.\n")
print(f"  {'pai era':<18}{'chega a A':>11}{'contra a média':>17}{'contra filho de profissional':>30}")
for g in sorted(GRUPOS, key=lambda x: -P_A[x]):
    print(f"  {g:<18}{pct(P_A[g]):>11}{vir(P_A[g] / p_a_total, 2) + '×':>17}"
          f"{vir(P_A[g] / P_A['Profissionais'], 2) + '×':>30}")
razao_extremos = P_A["Profissionais"] / P_A["Agrícolas"]
print(f"\n  Filho de profissional contra filho de agricultor: {vir(razao_extremos)} vezes mais chance.")
print(f"  Casa simples (produção, serviços, vendas, campo): entre {pct(P_A['Agrícolas'])} e "
      f"{pct(P_A['Vendedores'])}.")

# ── CAMADA 2 · subir, ficar, descer, por origem ──
print("\n" + "=" * 72)
print("  CAMADA 2 · SUBIR, FICAR OU DESCER, POR ORIGEM")
print("=" * 72)
def movimento(pai):
    o = ORDEM[ESTRATO[pai]]
    sobe = desce = fica = 0.0
    for filho in GRUPOS:
        e = ESTRATO[filho]
        if e == "FA":
            continue
        d = ORDEM[e]
        if d < o: sobe += M.loc[pai, filho]
        elif d > o: desce += M.loc[pai, filho]
        else: fica += M.loc[pai, filho]
    fa = M.loc[pai, "Forças Armadas"]
    return sobe, fica, desce, fa
print(f"\n  {'pai era':<18}{'sobe':>8}{'fica':>8}{'desce':>8}{'FA':>6}")
for g in GRUPOS[:-1]:
    s, f, d_, fa = movimento(g)
    print(f"  {g:<18}{pct(s):>8}{pct(f):>8}{pct(d_):>8}{pct(fa):>6}")
print(f"\n  Filho de agricultor não tem como descer, e sobe em {pct(movimento('Agrícolas')[0])}.")
print(f"  Filho de dirigente não tem como subir, e desce em {pct(movimento('Dirigentes')[2])}.")
print(f"  Os totais do IBGE ({pct(ASCENDENTE)} sobem) ponderam cada linha pelo número de")
print(f"  pais em cada grupo, que não está publicado; por isso não os recalculo aqui.")

# ── CAMADA 3 · a cadeia: quantas geracoes ──
# Hipotese declarada: a matriz vale igual para a proxima geracao (estacionariedade).
# E uma hipotese forte e serve para ler a escada, nao para prever 2050.
print("\n" + "=" * 72)
print("  CAMADA 3 · SE A MATRIZ VALESSE PARA SEMPRE: QUANTAS GERAÇÕES")
print("=" * 72)
Pm = M.values
def geracoes(origem, n):
    v = np.zeros(len(GRUPOS)); v[GRUPOS.index(origem)] = 1.0
    out = []
    for _ in range(n):
        v = v @ Pm
        out.append(v[:2].sum())
    return out
curvas = {g: geracoes(g, 6) for g in ["Agrícolas", "Produção", "Serviços", "Profissionais"]}
# distribuicao estacionaria
w, vl = np.linalg.eig(Pm.T)
est = np.real(vl[:, np.argmin(np.abs(w - 1))]); est = est / est.sum()
p_a_est = est[:2].sum()
print(f"\n  {'geração':>9}{'pai agrícola':>15}{'pai produção':>15}{'pai serviços':>15}{'pai profissional':>18}")
for i in range(6):
    print(f"  {i + 1:>9}{pct(curvas['Agrícolas'][i]):>15}{pct(curvas['Produção'][i]):>15}"
          f"{pct(curvas['Serviços'][i]):>15}{pct(curvas['Profissionais'][i]):>18}")
print(f"\n  Limite da cadeia (estado estacionário): {pct(p_a_est)} em A, contra {pct(p_a_total)}")
print(f"  observados hoje. O limite é maior porque a matriz carrega a mudança da economia")
print(f"  entre a geração dos pais e 2014 (mais vagas em A); projetá-la supõe que essa")
print(f"  mudança continue no mesmo ritmo. Por isso a curva é leitura da escada, não previsão.")
print(f"  O que ela mostra: a origem deixa de pesar em três gerações. A chance de uma")
print(f"  linhagem agrícola chegar a A vai de {pct(curvas['Agrícolas'][0])} para "
      f"{pct(curvas['Agrícolas'][2])} na terceira, e a de uma linhagem de profissionais")
print(f"  cai de {pct(curvas['Profissionais'][0])} para {pct(curvas['Profissionais'][2])}.")

# ── CAMADA 4 · a escola como degrau ──
print("\n" + "=" * 72)
print("  CAMADA 4 · O DEGRAU DA ESCOLA")
print("=" * 72)
print(f"\n  Filhos com ensino superior completo, segundo a instrução do pai:")
for k, v in SUPERIOR.items():
    print(f"    pai com {k:<24}{pct(v):>8}")
print(f"\n  De pai sem instrução a pai com superior: {vir(SUPERIOR['superior completo'] / SUPERIOR['sem instrução'])} vezes.")
print(f"  Cada degrau do pai vale, em média, "
      f"{vir((SUPERIOR['superior completo'] - SUPERIOR['sem instrução']) / 6 * 100)} pontos percentuais no filho.")

# ── CAMADA 5 · mil criancas ──
print("\n" + "=" * 72)
print("  CAMADA 5 · MIL CRIANÇAS DE CADA ORIGEM, SORTEADAS PELA MATRIZ")
print("=" * 72)
N_CRIANCAS, N_SIM = 1000, 20_000
print(f"\n  Em {mil(N_CRIANCAS)} crianças de cada origem, quantas chegam a A "
      f"({mil(N_SIM)} turmas sorteadas, semente 42):\n")
sim_res = {}
for g in ["Agrícolas", "Produção", "Serviços", "Vendedores", "Profissionais"]:
    cont = ger.binomial(N_CRIANCAS, P_A[g], size=N_SIM)
    sim_res[g] = cont
    lo, hi = np.percentile(cont, [2.5, 97.5])
    print(f"    pai {g:<14}{int(np.mean(cont)):>5} crianças   [{int(lo)}; {int(hi)}]")
print(f"\n  Isso é a variação de uma turma de mil, não a incerteza da estimativa do IBGE:")
print(f"  o IBGE não publica o número de entrevistados por célula, então o intervalo")
print(f"  de credibilidade de cada proporção não pode ser calculado com honestidade.")
print(f"  Com 58 mil entrevistas, ele é estreito nas células grandes e desconhecido nas pequenas.")
print("=" * 72)

In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · a escada ══
fig1, ax1 = plt.subplots(figsize=(11, 6.2))
ordem_g = sorted(GRUPOS[:-1], key=lambda x: P_A[x])
vals = [P_A[g] * 100 for g in ordem_g]
cores = [VERMELHO if ESTRATO[g] in ("F", "D", "E") else (DOURADO if ESTRATO[g] in ("B", "C") else VERDE)
         for g in ordem_g]
ax1.barh(ordem_g, vals, color=cores, alpha=0.88, height=0.62)
for i, v in enumerate(vals):
    ax1.text(v + 0.8, i, f"{vir(v)}%", va="center", fontsize=12, fontweight="bold")
ax1.axvline(p_a_total * 100, color="#333333", lw=1.3, ls="--")
ax1.text(p_a_total * 100 + 0.8, len(ordem_g) - 0.55, f"média de hoje: {pct(p_a_total)}", fontsize=10, color="#333333")
ax1.set_ylim(-0.6, len(ordem_g) - 0.1)
ax1.set_xlim(0, 68)
ax1.set_xlabel("filhos que estão hoje no estrato A, dirigentes e profissionais (%)")
ax1.set_title("A escada: a chance de chegar ao topo depende de onde o pai estava\n"
              "PNAD 2014, ocupação do pai quando a pessoa tinha 15 anos · vermelho: produção, serviços, vendas e campo",
              fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: IBGE, PNAD 2014, suplemento de mobilidade sócio-ocupacional · percentuais publicados"
            "  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-120-grafico-01-a-escada.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · a matriz ══
fig2, ax2 = plt.subplots(figsize=(11.5, 8.5))
sub = M.loc[GRUPOS[:-1], GRUPOS[:-1]] * 100
im = ax2.imshow(sub.values, cmap="Reds", vmin=0, vmax=50)
ax2.set_xticks(range(len(sub.columns))); ax2.set_xticklabels(sub.columns, rotation=35, ha="right", fontsize=10)
ax2.set_yticks(range(len(sub.index))); ax2.set_yticklabels(sub.index, fontsize=10)
for i in range(sub.shape[0]):
    for j in range(sub.shape[1]):
        v = sub.values[i, j]
        ax2.text(j, i, f"{vir(v)}", ha="center", va="center", fontsize=9.5,
                 color="white" if v > 28 else "#222222", fontweight="bold" if i == j else "normal")
ax2.set_xlabel("o que a pessoa faz hoje", fontsize=11)
ax2.set_ylabel("o que o pai fazia quando ela tinha 15 anos", fontsize=11)
ax2.grid(False)
ax2.set_title("De onde se parte, aonde se chega: a matriz inteira, em %\n"
              "a diagonal é ficar no grupo do pai; o canto superior esquerdo é a bancada herdada",
              fontsize=12.5, pad=14)
cb = plt.colorbar(im, ax=ax2, fraction=0.035, pad=0.02); cb.set_label("% dos filhos", fontsize=10)
plt.figtext(0.5, 0.005, "Fonte: IBGE, PNAD 2014 · linhas somam 100 · Forças Armadas omitidas (0,2% a 6,9%)"
            "  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-120-grafico-02-a-matriz.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · as geracoes ══
fig3, ax3 = plt.subplots(figsize=(11, 6))
cor_g = {"Agrícolas": VERMELHO, "Produção": DOURADO, "Serviços": "#8a6f1f", "Profissionais": VERDE}
for g, c in curvas.items():
    ax3.plot(range(1, 7), np.array(c) * 100, marker="o", lw=2.6, color=cor_g[g], label=f"pai {g.lower()}")
ax3.axhline(p_a_est * 100, color="#333333", lw=1.2, ls="--")
ax3.text(1.0, p_a_est * 100 + 0.8, f"limite da cadeia: {pct(p_a_est)} (hoje: {pct(p_a_total)})", fontsize=10, color="#333333", ha="left")
ax3.set_xlabel("geração (1 = o filho; 2 = o neto; 3 = o bisneto)")
ax3.set_ylabel("chance de estar no estrato A (%)")
ax3.set_xticks(range(1, 7))
ax3.legend(fontsize=10, loc="lower right")
ax3.set_title("Se a matriz valesse para sempre, a origem deixaria de pesar em três gerações\n"
              "cadeia de Markov sobre a matriz de 2014 · hipótese de estacionariedade, declarada",
              fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Conta própria sobre a tabela do IBGE · não é previsão, é a leitura da escada no limite"
            "  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-120-grafico-03-as-geracoes.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · a escola ══
fig4, ax4 = plt.subplots(figsize=(11, 6))
ks = list(SUPERIOR.index); vs = SUPERIOR.values * 100
ax4.bar(range(len(ks)), vs, color=[VERMELHO, VERMELHO, DOURADO, DOURADO, DOURADO, VERDE, VERDE], alpha=0.88, width=0.62)
for i, v in enumerate(vs):
    ax4.text(i, v + 1.2, f"{vir(v)}%", ha="center", fontsize=12, fontweight="bold")
ax4.set_xticks(range(len(ks))); ax4.set_xticklabels([k.replace(" ", "\n", 1) for k in ks], fontsize=10)
ax4.set_ylim(0, 80)
ax4.set_ylabel("filhos com ensino superior completo (%)")
ax4.set_xlabel("instrução do pai")
ax4.set_title(f"O degrau da escola: de 4,0% a 69,1%, {vir(SUPERIOR['superior completo'] / SUPERIOR['sem instrução'])} vezes\n"
              "pessoas de 25 anos ou mais que moravam com o pai aos 15 · PNAD 2014",
              fontsize=12.5, pad=14)
plt.figtext(0.5, 0.005, "Fonte: IBGE, PNAD 2014, suplemento de mobilidade sócio-ocupacional"
            "  |  #365Probabilidades", ha="center", fontsize=8.5, color="gray")
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig("dia-120-grafico-04-a-escola.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**47,4% das pessoas chegaram a um estrato acima do pai.** Um terço ficou no mesmo.
17,2% desceram. Essa é a resposta da pergunta do título, na média do país.

Dentro dela, a escada: uma criança cujo pai trabalhava no campo tem 7,5% de chance de
chegar ao estrato de dirigentes e profissionais. A filha de um profissional tem 59,0%.
Quase oito vezes. Entre as duas, as casas simples da cidade: filhos de trabalhadores da
produção, dos serviços e do comércio chegam lá em 13% a 20%.

A escola é o degrau que mais pesa. De pai sem instrução, 4,0% dos filhos completam o
ensino superior. De pai com superior, 69,1%. Dezessete vezes.

Se a matriz de 2014 valesse para sempre, a origem deixaria de pesar em três gerações.
Não é previsão. É a leitura da escada no limite.

O que a tabela não mede é o que a memória faz com o caminho. A aspiração sobe junto com
a renda, e o sonho realizado vira régua. A bancada deixa de ser a bancada que faltava e
vira a bancada que existe. Foi assim que eu passei anos escovando os dentes num sonho
sem saber.

*De onde você partiu, e o que você já realizou que hoje chama de normal?*

---

### ⚠️ Limitações do Modelo

- Os percentuais da matriz foram lidos por extração do PDF da divulgação do IBGE
  (16/11/2016). As linhas somam 100 e o padrão interno é coerente, mas a ordem das
  colunas foi reconstruída e **precisa ser conferida contra o PDF antes do commit**.
- O IBGE publica proporções, não contagens por célula. Sem o número de entrevistados em
  cada combinação pai × filho, não há intervalo de credibilidade honesto por célula. A
  simulação da camada 5 mostra a variação de uma turma de mil, não a incerteza da
  estimativa.
- "Casa simples" e "bancada" são traduções minhas dos estratos do IBGE (origem fora de
  A; destino em A). A tabela mede ocupação, não renda, patrimônio nem reforma de banheiro.
- A cadeia de Markov supõe que a matriz de 2014 vale para as próximas gerações. É
  hipótese forte, declarada, e serve para ler a escada, não para prever.
- Os totais de 47,4% / 33,4% / 17,2% são do IBGE; não os recalculo porque a
  distribuição dos pais por grupo não está na divulgação.
- A tabela é de 2014. Não há suplemento mais recente da PNAD com a mesma pergunta.
- O mecanismo da aspiração (Easterlin 2001) entra como conceito, sem número; a
  quantificação de Stutzer 2004 não foi conferida no PDF e ficou de fora.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
